In [ ]:
!pip install ultralytics

In [1]:
from PIL import Image
from ultralytics import YOLO

In [31]:
def get_cropped_faces(filenames):
    """
    filenames: list of filename

    Return:
        [[file1_cropped_face1, file1_cropped_face2, ...], [file2_cropped_face1, ...], ...]
    """
    yolov8_animeface = YOLO('Model/yolov8x6_animeface.pt')
    results = yolov8_animeface.predict(filenames, save=False, conf=0.3, iou=0.5)
    bounding_boxes = get_bounding_boxes(results)
    cropped_faces = []
    for filename, boxes in zip(filenames, bounding_boxes):
        file_cropped_faces = []
        for box in boxes:
            cropped = get_cropped(filename, box['top_left'], box['bottom_right'])
            file_cropped_faces.append(cropped)
        cropped_faces.append(file_cropped_faces)
    return cropped_faces

# Helper Functions
def get_bounding_boxes(results):
    """
    Input
        results: list of list of detected output
    
    Return
        List of dicts of 
    """
    out = []
    for result in results:
        boxes = result.boxes
        bounding_boxes = []
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()  # top-left and bottom-right corners
            confidence = box.conf[0].item()
            
            bound_box = {'top_left': (x1, y1), 'bottom_right': (x2, y2)}
            bounding_boxes.append(bound_box)
        out.append(bounding_boxes)
    return out


def get_cropped(filename, top_left, bottom_right):
    """
    Input
        filename: image file name (str)
        top_left: location of top-left corner (len-2 tuple: (x1, y1))
        bottom_right: location of bottom_right corner (len-2 tuple: (x2, y2))
    
    Return
        cropped img
    """
    img = Image.open(filename)

    # Pillow crop takes (left, upper, right, lower)
    cropped = img.crop((top_left[0], top_left[1], bottom_right[0], bottom_right[1]))
    return cropped

In [ ]:
get_cropped_faces(['Manga109/images/AisazuNihaIrarenai/059.jpg'])[0][0]

In [ ]:
for result in results:
    boxes = result.boxes
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()  # top-left and bottom-right corners
        confidence = box.conf[0].item()
        
        print(f"Face detected:")
        print(f"  Top-left:     ({x1:.1f}, {y1:.1f})")
        print(f"  Bottom-right: ({x2:.1f}, {y2:.1f})")
        print(f"  Width:        {x2 - x1:.1f}px")
        print(f"  Height:       {y2 - y1:.1f}px")
        print(f"  Confidence:   {confidence:.2f}")
        print()